In [1]:
! python --version

Python 3.12.13


Домашнее задание по генерации речи
Few-shot voice cloning модели XTTS
Cтатья: XTTS: a Massively Multilingual Zero-Shot Text-to-Speech Model.

github: https://github.com/coqui-ai/TTS

документация: https://docs.coqui.ai/en/dev/models/xtts.html

demo: https://edresson.github.io/XTTS/

На семинаре были примеры спикеров, на которых zero-shot генерация по одной референсной записи показывала не очень высокий уровень похожести голоса. Альтернативным сценарием явялется дообучение модели на данных такого диктора, которое и предлагается выполнить в этом домашнем задании.

##1. Настройка окружения
Установите необходимые пакеты, следуя документации (установка через pip либо скачивание репозитория с github).

In [2]:
!git clone https://github.com/idiap/coqui-ai-TTS && cd coqui-ai-TTS && pip install -e .

Cloning into 'coqui-ai-TTS'...
remote: Enumerating objects: 35779, done.
remote: Total 35779 (delta 0), reused 0 (delta 0), pack-reused 35779 (from 1)
Receiving objects: 100% (35779/35779), 138.12 MiB | 14.66 MiB/s, done.
Resolving deltas: 100% (25937/25937), done.
Obtaining file:///content/coqui-ai-TTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 18.6 MB/s eta 0:00:00
 

In [3]:
!pip install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 64.0 MB/s eta 0:00:00


In [4]:
%cd coqui-ai-TTS

/content/coqui-ai-TTS


In [5]:
%pwd

'/content/coqui-ai-TTS'

In [6]:
import os
from IPython.display import Audio, display

In [7]:
from TTS.api import TTS
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")

You must confirm the following:
  "I have purchased a commercial license from Coqui: licensing@coqui.ai"
  "Otherwise, I agree to the terms of the non-commercial CPML: https://tts-hub.github.io/cpml" - [y/n]
   > y


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [8]:
out_dir = 'generated_samples/'
!mkdir $out_dir

##2. Сбор и подготовка данных
2.1. подберите голос персонажа/актера/..., который плохо клонируется в zero-shot формате.

2.2. соберите как минимум 5-10 минут его голоса

**Для клонирования был выбран гоглос актера озвучки Рогволда Суховерко в зрелом возрасте(ближе к старости он озвучивал Гендальфа и Хагрида)**

 **Далее был выбран 15ти минутный фрагмент из аудикниги, начитанный Рогволдом Суховерко, и подготовлены данные для обучения - фрагменты в среднем по 4,36 секунды**

In [12]:
import pandas as pd
import soundfile as sf

In [10]:
import os
from pydub import AudioSegment
from faster_whisper import WhisperModel

# --- НАСТРОЙКИ ---
INPUT_AUDIO = "/content/Rogvold_Suhoverko.mp3"  # Путь к вашему исходному файлу
OUTPUT_DIR = "dataset"           # Папка, куда сохранится датасет
LANGUAGE = "ru"

os.makedirs(f"{OUTPUT_DIR}/wavs", exist_ok=True)

# 1. Загрузка модели Whisper для транскрибации и сегментации
model = WhisperModel("large-v3", device="cuda", compute_type="float16") # или device="cpu"

# 2. Запуск распознавания
segments, info = model.transcribe(INPUT_AUDIO, language=LANGUAGE, beam_size=5)

# 3. Нарезка и формирование metadata.csv
full_audio = AudioSegment.from_file(INPUT_AUDIO)

with open(f"{OUTPUT_DIR}/metadata.csv", "w", encoding="utf-8") as f:
    for i, segment in enumerate(segments):
        # Ограничиваем длину сегмента (от 2 до 10 секунд — идеал для XTTS)
        duration = segment.end - segment.start
        if 2.0 < duration < 12.0:
            start_ms = segment.start * 1000
            end_ms = segment.end * 1000

            # Извлекаем кусок аудио
            chunk = full_audio[start_ms:end_ms]

            file_name = f"segment_{i:04d}"
            file_path = f"{OUTPUT_DIR}/wavs/{file_name}.wav"

            # Сохраняем в нужном формате (wav, mono, 22050Hz)
            chunk.set_frame_rate(22050).set_channels(1).export(file_path, format="wav")

            # Записываем в метаданные (формат: file|text|text)
            text = segment.text.strip()
            f.write(f"{file_name}|{text}|{text}\n")

print(f"Готово! Данные сохранены в папку: {OUTPUT_DIR}")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Готово! Данные сохранены в папку: dataset


In [ ]:
import os
import wave

# Путь к папке с нарезанными файлами
wav_dir = "dataset/wavs"
wav_files = [f for f in os.listdir(wav_dir) if f.endswith('.wav')]

total_duration = 0

for wav in wav_files:
    with wave.open(os.path.join(wav_dir, wav), 'rb') as f:
        # Длительность = количество кадров / частоту дискретизации
        frames = f.getnframes()
        rate = f.getframerate()
        total_duration += frames / float(rate)

count = len(wav_files)
avg_duration = total_duration / count if count > 0 else 0

print(f'Количество аудиозаписей: {count}')
print(f'Общая длительность: {total_duration:.2f} сек ({total_duration/60:.2f} мин), средняя длительность записи: {avg_duration:.2f} сек')

Количество аудиозаписей: 165
Общая длительность: 719.22 сек (11.99 мин), средняя длительность записи: 4.36 сек


**Результат анализа полученного датасета:**

Количество аудиозаписей: 165
Общая длительность: 719.22 сек (11.99 мин), средняя длительность записи: 4.36 сек

**Хорошо подходит для дообучения XTTS**

**Помсотрим на файл metadata.csv**

In [ ]:
pd.read_csv('/content/coqui-ai-TTS/dataset/metadata.csv', header=None, sep='|')

,0,1,2
0,segment_0000,Среди погибших не значатся.,Среди погибших не значатся.
1,segment_0001,Старшина спецлагеря Азаров,Старшина спецлагеря Азаров
2,segment_0002,Подполковник Бурсов с майором Агинским уже тре...,Подполковник Бурсов с майором Агинским уже тре...
3,segment_0003,"Им известно, что это карантин, и что во второй...","Им известно, что это карантин, и что во второй..."
4,segment_0004,Их привел сюда унтерфедфебель Краус.,Их привел сюда унтерфедфебель Краус.
...,...,...,...
160,segment_0200,"Мы лишь предполагаем, что атомы в их молекулах...","Мы лишь предполагаем, что атомы в их молекулах..."
161,segment_0201,"Нечто вроде равновесия карандаша, поставленног...","Нечто вроде равновесия карандаша, поставленног..."
162,segment_0202,Беспокоит Бурсова и старший лейтенант Сердюк.,Беспокоит Бурсова и старший лейтенант Сердюк.
163,segment_0203,"Он хоть и был неплохим полковым инженером, но ...","Он хоть и был неплохим полковым инженером, но ..."


**Пример аудизаписи из датесета для дообучения:**

In [ ]:
display(Audio('/content/coqui-ai-TTS/dataset/wavs/segment_0000.wav'))

**Напишем код для оценки похожести голосов:**

In [ ]:
from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector
import torch
import torchaudio
import numpy as np

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-sv')
model = WavLMForXVector.from_pretrained('microsoft/wavlm-base-sv')

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/405M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

In [ ]:
def load_audio(audio_path):
    a, sr = torchaudio.load(audio_path)
    if sr != 16000:
        resample_rate = 16000
        resampler = torchaudio.transforms.Resample(sr, resample_rate, dtype=a.dtype)
        resampled_waveform = resampler(a)
        a = resampled_waveform
    return a

def extract_embedding(audio_path):
    audio = load_audio(audio_path)
    inputs = feature_extractor(audio[0], sampling_rate=16000, return_tensors="pt")
    embeddings = model(**inputs).embeddings
    embeddings = torch.nn.functional.normalize(embeddings, dim=-1).cpu()
    return embeddings[0]

def cosine_similarity(emb1, emb2):
    cosine_sim = torch.nn.CosineSimilarity(dim=-1)
    similarity = cosine_sim(emb1, emb2)
    return similarity

**Сравним похожесть одного из файлов с оригинальным голосом Рогволда с другими файлами c оригинальным голосом Рогволда.**

In [ ]:
# 1. Задаем пути
reference_file = '/content/coqui-ai-TTS/dataset/wavs/segment_0000.wav'

# Формируем список из 10 файлов циклом (от 0001 до 0010)
candidate_files = [f'/content/coqui-ai-TTS/dataset/wavs/segment_{i:04d}.wav' for i in range(1, 11)]

# 2. Извлекаем эмбеддинг эталона
ref_emb = extract_embedding(reference_file)

scores = []

print(f"{'Файл':<60} | {'Похожесть':<10}")
print("-" * 75)

# 3. Сравниваем в цикле
for seg_path in candidate_files:
    try:
        seg_emb = extract_embedding(seg_path)
        score = cosine_similarity(ref_emb, seg_emb).item()
        scores.append(score)
        print(f"{seg_path:<60} | {score:.4f}")

    except Exception as e:
        print(f"Ошибка в файле {seg_path}: {e}")

# 4. Вывод среднего значения
if scores:
    mean_score = sum(scores) / len(scores)
    print("-" * 75)
    print(f"{'СРЕДНЕЕ ЗНАЧЕНИЕ:':<60} | {mean_score:.4f}")



Файл                                                         | Похожесть 
---------------------------------------------------------------------------
/content/coqui-ai-TTS/dataset/wavs/segment_0001.wav          | 0.9467
/content/coqui-ai-TTS/dataset/wavs/segment_0002.wav          | 0.9327
/content/coqui-ai-TTS/dataset/wavs/segment_0003.wav          | 0.9681
/content/coqui-ai-TTS/dataset/wavs/segment_0004.wav          | 0.8868
/content/coqui-ai-TTS/dataset/wavs/segment_0005.wav          | 0.9435
Ошибка в файле /content/coqui-ai-TTS/dataset/wavs/segment_0006.wav: Failed to create AudioDecoder for /content/coqui-ai-TTS/dataset/wavs/segment_0006.wav: Could not open input file: /content/coqui-ai-TTS/dataset/wavs/segment_0006.wav No such file or directory
/content/coqui-ai-TTS/dataset/wavs/segment_0007.wav          | 0.9569
/content/coqui-ai-TTS/dataset/wavs/segment_0008.wav          | 0.9422
/content/coqui-ai-TTS/dataset/wavs/segment_0009.wav          | 0.9605
/content/coqui-ai-TTS/dataset/

**СРЕДНЕЕ ЗНАЧЕНИЕ похожести: 0.9412**

**Предложения для синтеза:**

In [ ]:
sentences = [
    'Он был красив, смел и приятен, бесцеремонен, весел и добр.',
    'С этой вспышкой, наконец.',
    'Он провел рукой по глазам, слегка поморщившись.',
    'Мне придется послать в город. На это последовал единодушный стон и множество упреков; после чего он, в своей отрешенной манере, объяснился.',
    'Я храню это ЗДЕСЬ — он постучал по сердцу. Я никогда этого не терял.',
    'Он не обращал на нее внимания; он смотрел на меня, но так, будто вместо меня видел то, о чем говорил.',
    'Если один ребенок затягивает гайку еще на один оборот, что вы скажете о ДВУХ детях?',
    'Также это вексель на будущее, который иногда оплачивается, но чаще продлевается.',
    'Мари всплеснула руками и вскочила со своего места.',
    'Старики в горах сажают липы, чтобы очистить лес.'
]


In [ ]:
out_dir_sim_test = os.path.join(out_dir, 'sim_test')
os.makedirs(out_dir_sim_test, exist_ok=True)

In [ ]:
os.makedirs("reference_audios", exist_ok=True)

**Сгенерируем десять предложений на одном примере:**

In [ ]:
lang='ru'

for i, sent in enumerate(sentences):
    out_file = f'{out_dir_sim_test}/{i}_male.wav'
    tts.tts_to_file(text=sentences[i],
                   file_path=out_file,
                   speaker_wav='/content/coqui-ai-TTS/dataset/wavs/segment_0003.wav',
                   language=lang,
                   split_sentences=True
                   )

**Сравним похожесть одного из файлов с оригинальным голосом Рогволда с другими файлами c клонированным голосом Рогволда.**

In [ ]:
# 1. Задаем пути
reference_file = '/content/coqui-ai-TTS/dataset/wavs/segment_0000.wav'
candidate_files = [
    '/content/coqui-ai-TTS/generated_samples/sim_test/0_male.wav',
    '/content/coqui-ai-TTS/generated_samples/sim_test/1_male.wav',
    '/content/coqui-ai-TTS/generated_samples/sim_test/2_male.wav',
    '/content/coqui-ai-TTS/generated_samples/sim_test/3_male.wav',
    '/content/coqui-ai-TTS/generated_samples/sim_test/4_male.wav',
    '/content/coqui-ai-TTS/generated_samples/sim_test/5_male.wav',
    '/content/coqui-ai-TTS/generated_samples/sim_test/6_male.wav',
    '/content/coqui-ai-TTS/generated_samples/sim_test/7_male.wav',
    '/content/coqui-ai-TTS/generated_samples/sim_test/8_male.wav',
    '/content/coqui-ai-TTS/generated_samples/sim_test/9_male.wav'
]

# 2. Извлекаем эмбеддинг эталона
ref_emb = extract_embedding(reference_file)

scores = []

print(f"{'Файл':<60} | {'Похожесть':<10}")
print("-" * 75)

# 3. Сравниваем в цикле
for seg_path in candidate_files:
    try:
        seg_emb = extract_embedding(seg_path)
        score = cosine_similarity(ref_emb, seg_emb).item()
        scores.append(score)
        print(f"{seg_path:<60} | {score:.4f}")

    except Exception as e:
        print(f"Ошибка в файле {seg_path}: {e}")

# 4. Вывод среднего значения
if scores:
    mean_score = sum(scores) / len(scores)
    print("-" * 75)
    print(f"{'СРЕДНЕЕ ЗНАЧЕНИЕ:':<60} | {mean_score:.4f}")



Файл                                                         | Похожесть 
---------------------------------------------------------------------------
/content/coqui-ai-TTS/generated_samples/sim_test/0_male.wav  | 0.8887
/content/coqui-ai-TTS/generated_samples/sim_test/1_male.wav  | 0.8552
/content/coqui-ai-TTS/generated_samples/sim_test/2_male.wav  | 0.7842
/content/coqui-ai-TTS/generated_samples/sim_test/3_male.wav  | 0.8200
/content/coqui-ai-TTS/generated_samples/sim_test/4_male.wav  | 0.9093
/content/coqui-ai-TTS/generated_samples/sim_test/5_male.wav  | 0.8637
/content/coqui-ai-TTS/generated_samples/sim_test/6_male.wav  | 0.8737
/content/coqui-ai-TTS/generated_samples/sim_test/7_male.wav  | 0.8813
/content/coqui-ai-TTS/generated_samples/sim_test/8_male.wav  | 0.8417
/content/coqui-ai-TTS/generated_samples/sim_test/9_male.wav  | 0.8981
---------------------------------------------------------------------------
СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.8616


**СРЕДНЕЕ ЗНАЧЕНИЕ похожести: 0.8616 против  0.9412 у оригинальных данных**

**Прослушаем сгенерированное аудио:**

In [ ]:
display(Audio('/content/coqui-ai-TTS/generated_samples/sim_test/3_male.wav'))

**Похожесть явно слышна но голос довольно роботизированный**

##3. **Дообучение**
3.1. в документации есть описание того, как происходит дообучение, прочтите необходимую информацию

3.2. Ответьте на вопрос: "Какая часть модели обновляется во время дообучения?"

**В процессе дообучения (файнтюнинга) XTTS чаще всего обновляется GPT Encoder: Модель учится лучше предсказывать последовательность латентных кодов, которые соответствуют тембру и интонации конкретного человека. Это и позволяет «подхватить» уникальную манеру речи.**

**Далее дообучим модель на десяти минутах датасета аудио подогтовленного ранее.**

In [ ]:
os.mkdir('generated_samples/checkpoints')

In [ ]:
import os

from trainer import Trainer, TrainerArgs

from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
from TTS.tts.models.xtts import XttsAudioConfig
from TTS.utils.manage import ModelManager

**Зададим параметры дообучения, как у авторов XTTS из их репозитория:**

In [ ]:
# Logging parameters
RUN_NAME = "GPT_XTTS_v2.0_LJSpeech_FT"
PROJECT_NAME = "XTTS_trainer"
DASHBOARD_LOGGER = "tensorboard"
LOGGER_URI = None

# Set here the path that the checkpoints will be saved. Default: ./run/training/
OUT_PATH = '/content/coqui-ai-TTS/generated_samples/checkpoints'

# Training Parameters
OPTIMIZER_WD_ONLY_ON_WEIGHTS = True  # for multi-gpu training please make it False
START_WITH_EVAL = False  # if True it will star with evaluation
BATCH_SIZE = 2  # set here the batch size
GRAD_ACUMM_STEPS = 8  # set here the grad accumulation steps
# Note: we recommend that BATCH_SIZE * GRAD_ACUMM_STEPS need to be at least 252 for more efficient training. You can increase/decrease BATCH_SIZE but then set GRAD_ACUMM_STEPS accordingly.
EPOCHS = 150
# Define here the dataset that you want to use for the fine-tuning on.
config_dataset = BaseDatasetConfig(
    formatter="ljspeech",
    dataset_name="ljspeech",
    path="/content/coqui-ai-TTS/dataset",
    meta_file_train="/content/coqui-ai-TTS/dataset/metadata.csv",
    language="ru",
)

# Add here the configs of the datasets
DATASETS_CONFIG_LIST = [config_dataset]

# Define the path where XTTS v2.0.1 files will be downloaded
CHECKPOINTS_OUT_PATH = os.path.join(OUT_PATH, "XTTS_v2.0_original_model_files/")
os.makedirs(CHECKPOINTS_OUT_PATH, exist_ok=True)


# DVAE files
DVAE_CHECKPOINT_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth"
MEL_NORM_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/mel_stats.pth"

# Set the path to the downloaded files
DVAE_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(DVAE_CHECKPOINT_LINK))
MEL_NORM_FILE = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(MEL_NORM_LINK))

# Download XTTS v2.0 checkpoint if needed
TOKENIZER_FILE_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json"
XTTS_CHECKPOINT_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/model.pth"

# XTTS transfer learning parameters: You we need to provide the paths of XTTS model checkpoint that you want to do the fine tuning.
TOKENIZER_FILE = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(TOKENIZER_FILE_LINK))  # vocab.json file
XTTS_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(XTTS_CHECKPOINT_LINK))  # model.pth file

# Training sentences generations
SPEAKER_REFERENCE = [
    "/content/coqui-ai-TTS/dataset/wavs/segment_0003.wav"  # speaker reference to be used in training test sentences
]
LANGUAGE = config_dataset.language

**Напишем функцию дообучения:**

In [ ]:
# download DVAE files if needed
if not os.path.isfile(DVAE_CHECKPOINT) or not os.path.isfile(MEL_NORM_FILE):
    print(" > Downloading DVAE files!")
    ModelManager._download_model_files([MEL_NORM_LINK, DVAE_CHECKPOINT_LINK], CHECKPOINTS_OUT_PATH, progress_bar=True)




# download XTTS v2.0 files if needed
if not os.path.isfile(TOKENIZER_FILE) or not os.path.isfile(XTTS_CHECKPOINT):
    print(" > Downloading XTTS v2.0 files!")
    ModelManager._download_model_files(
        [TOKENIZER_FILE_LINK, XTTS_CHECKPOINT_LINK], CHECKPOINTS_OUT_PATH, progress_bar=True
    )


def main():
    # init args and config
    model_args = GPTArgs(
        max_conditioning_length=132300,  # 6 secs
        min_conditioning_length=66150,  # 3 secs
        debug_loading_failures=False,
        max_wav_length=255995,  # ~11.6 seconds
        max_text_length=200,
        mel_norm_file=MEL_NORM_FILE,
        dvae_checkpoint=DVAE_CHECKPOINT,
        xtts_checkpoint=XTTS_CHECKPOINT,  # checkpoint path of the model that you want to fine-tune
        tokenizer_file=TOKENIZER_FILE,
        gpt_num_audio_tokens=1026,
        gpt_start_audio_token=1024,
        gpt_stop_audio_token=1025,
        gpt_use_masking_gt_prompt_approach=True,
        gpt_use_perceiver_resampler=True,
    )
    # define audio config
    audio_config = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000)
    # training parameters config
    config = GPTTrainerConfig(
        output_path=OUT_PATH,
        model_args=model_args,
        datasets=[config_dataset],
        run_name=RUN_NAME,
        project_name=PROJECT_NAME,
        run_description="""
            GPT XTTS training
            """,
        dashboard_logger=DASHBOARD_LOGGER,
        logger_uri=LOGGER_URI,
        audio=audio_config,
        epochs = EPOCHS,
        batch_size=BATCH_SIZE,
        batch_group_size=48,
        eval_batch_size=BATCH_SIZE,
        num_loader_workers=4,
        eval_split_max_size=256,
        print_step=3,
        plot_step=10,
        log_model_step=20,
        save_step=100,
        save_n_checkpoints=2,
        save_checkpoints=False,
        # target_loss="loss",
        print_eval=False,
        run_eval=False,
        # Optimizer values like tortoise, pytorch implementation with modifications to not apply WD to non-weight parameters.
        optimizer="AdamW",
        optimizer_wd_only_on_weights=OPTIMIZER_WD_ONLY_ON_WEIGHTS,
        optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
        lr=5e-06,  # learning rate
        lr_scheduler="MultiStepLR",
        # it was adjusted accordly for the new step scheme
        lr_scheduler_params={"milestones": [2000, 6000, 12000], "gamma": 0.5, "last_epoch": -1},
        test_sentences=[
            {
                "text": "не придется послать в город. На это последовал единодушный стон и множество упреков.",
                "speaker_wav": SPEAKER_REFERENCE[0],
                "language": LANGUAGE,
            },
            {
                "text": "С каждой минутой голос звучит всё лучше и естественнее.",
                "speaker_wav": SPEAKER_REFERENCE[0],
                "language": LANGUAGE,
            },
        ],
    )

    # init the model from config
    model = GPTTrainer.init_from_config(config)

    # load training samples
    train_samples, eval_samples = load_tts_samples(
        config,
        eval_split=False
    )

    # init the trainer and 🚀
    trainer = Trainer(
        TrainerArgs(
            restore_path=None,  # xtts checkpoint is restored via xtts_checkpoint key so no need of restore it using Trainer restore_path parameter
            skip_train_epoch=False,
            start_with_eval=False,
            grad_accum_steps=GRAD_ACUMM_STEPS,
        ),
        config,
        output_path=OUT_PATH,
        model=model,
        train_samples=train_samples,
        eval_samples=[],
    )
    trainer.fit()

 > Downloading DVAE files!


100%|██████████| 1.07k/1.07k [00:00<00:00, 2.32MiB/s]
  1%|          | 1.95M/211M [00:00<00:32, 6.41MiB/s]

model.safetensors:   0%|          | 0.00/404M [00:00<?, ?B/s]

100%|██████████| 211M/211M [00:08<00:00, 24.0MiB/s]


 > Downloading XTTS v2.0 files!


0.00iB [00:00, ?iB/s]
100%|██████████| 1.87G/1.87G [00:17<00:00, 104MiB/s]


In [ ]:
ls

CITATION.cff        dockerfiles/        LICENSE.txt     recipes/
CODE_OF_CONDUCT.md  docs/               Makefile        reference_audios/
CONTRIBUTING.md     generated_samples/  notebooks/      scripts/
dataset/            hubconf.py          pyproject.toml  tests/
Dockerfile          images/             README.md       TTS/


In [ ]:
cd ..

/content


**Обучим на 50ти плюс эпохах:**

In [ ]:
main()

Выходные данные были обрезаны до нескольких последних строк (5000).
     | > loss_mel_ce: 2.431277275085449  (2.5797946300912398)
     | > loss: 0.30773988366127014  (0.3263258369679147)
     | > grad_norm: 0.0  (0.0)
     | > step_time: 0.1234  (0.12696848017104126)
     | > loader_time: 0.0096  (0.010159081601082011)


   --> TIME: 2026-05-06 14:03:25 -- STEP: 50/55 -- GLOBAL_STEP: 1506
     | > current_lr: 5e-06  (5.0000000000000055e-06)
     | > loss_text_ce: 0.030751749873161316  (0.03068410836160183)
     | > loss_mel_ce: 2.2221240997314453  (2.548395011425017)
     | > loss: 0.2816094756126404  (0.3223848873376846)
     | > step_time: 0.1002  (0.12535125255584711)
     | > loader_time: 0.0096  (0.010118446350097655)


   --> TIME: 2026-05-06 14:03:25 -- STEP: 53/55 -- GLOBAL_STEP: 1509
     | > current_lr: 5e-06  (5.000000000000006e-06)
     | > loss_text_ce: 0.03593141958117485  (0.030769370461128792)
     | > loss_mel_ce: 2.6955254077911377  (2.5582464213641174)
     | > loss:

**Инициализируем и загрузим дообученную модель XTTS из чекпойнта.**

In [ ]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# Add here the vocab file that you have used to train the model
TOKENIZER_PATH = TOKENIZER_FILE
# Add here the checkpoint that you want to do inference with
XTTS_CHECKPOINT = '/content/coqui-ai-TTS/generated_samples/checkpoints/GPT_XTTS_v2.0_LJSpeech_FT-May-06-2026_01+42PM-0000000/best_model.pth'

CONFIG_PATH = "/content/coqui-ai-TTS/generated_samples/checkpoints/GPT_XTTS_v2.0_LJSpeech_FT-May-06-2026_01+42PM-0000000/config.json"
# Add here the speaker reference
SPEAKER_REFERENCE = "/content/coqui-ai-TTS/dataset/wavs/segment_0003.wav"

# output wav path
OUTPUT_WAV_PATH = "xtts-ft.wav"

print("Loading model...")
config = XttsConfig()
config.load_json(CONFIG_PATH)
tts_ft = Xtts.init_from_config(config)
tts_ft.load_checkpoint(config, checkpoint_path=XTTS_CHECKPOINT, vocab_path=TOKENIZER_PATH, use_deepspeed=False)
tts_ft.cuda()


Loading model...


/tmp/ipykernel_36120/1314952334.py:19: UserWarning: Xtts.init_from_config(config) is deprecated and will be removed soon, just initialize with Xtts(config)
  tts_ft = Xtts.init_from_config(config)


Xtts(
  (gpt): GPT(
    (conditioning_encoder): ConditioningEncoder(
      (init): Conv1d(80, 1024, kernel_size=(1,), stride=(1,))
      (attn): Sequential(
        (0): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttentionLegacy()
          (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
        )
        (1): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttentionLegacy()
          (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
        )
        (2): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttentionLegacy()
          (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(

##**4. Оценка результатов**
4.1. Для каждой из 3х моделей (zero-shot, дообученная на 1 минуте, дообученная на всех данных) сгенерируйте 10 предложени

4.2. Для каждой из моделей оцените speaker similarity моделью WavLM-sv, как мы делали на семинаре

!!! в качестве референсной записи выберите 3 различных варианта и подсчитайте similarity:

этих записей друг с другом

этих записей со всеми сгенерированными

**Сгенерируем 10ть предложений:**

In [ ]:
import torch
import torchaudio
import os

# 1. Извлекаем характеристики голоса
print("Computing speaker latents...")
gpt_cond_latent, speaker_embedding = tts_ft.get_conditioning_latents(audio_path=[SPEAKER_REFERENCE])


# 2. Цикл генерации и сохранения в отдельные файлы
print("Inference started...")
for i, text in enumerate(sentences):
    # Формируем имя файла для каждого предложения
    file_path = f"sentence_{i}.wav"

    print(f"Generating {file_path}...")

    out = tts_ft.inference(
        text=text,
        language="ru",
        gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding,
        temperature=0.75,
    )

    # Сохраняем тензор в отдельный файл
    wav_tensor = torch.tensor(out["wav"]).unsqueeze(0)
    torchaudio.save(file_path, wav_tensor, 24000)

print("Все файлы успешно сохранены!")



Computing speaker latents...
Inference started...
Generating sentence_0.wav...
Generating sentence_1.wav...
Generating sentence_2.wav...
Generating sentence_3.wav...
Generating sentence_4.wav...
Generating sentence_5.wav...
Generating sentence_6.wav...
Generating sentence_7.wav...
Generating sentence_8.wav...
Generating sentence_9.wav...
Все файлы успешно сохранены!


### **Возьмем для референса три аудиозаписи:**

segment_0000.wav, segment_0001.wav, segment_0002.wav

Ранее мы уже смотрли похожесть segment_0000.wav c  segment_0001.wav и  segment_0002.wav, результаты соотвественно:

 0.9467 | 0.9327

### **Модель, обученная на десяти минутах:**

**Посчитаем похожесть с первым референсом segment_0000.wav**

In [ ]:
# 1. Задаем пути
reference_file = '/content/coqui-ai-TTS/dataset/wavs/segment_0000.wav'

# Автоматически создаем список путей от sentence_0.wav до sentence_9.wav
candidate_files = [f'/content/sentence_{i}.wav' for i in range(10)]

# 2. Извлекаем эмбеддинг эталона
ref_emb = extract_embedding(reference_file)

scores = []

print(f"{'Файл':<60} | {'Похожесть':<10}")
print("-" * 75)

# 3. Сравниваем в цикле
for seg_path in candidate_files:
    # Проверяем, существует ли файл, чтобы код не упал с ошибкой
    if not os.path.exists(seg_path):
        print(f"Файл не найден: {seg_path}")
        continue

    try:
        seg_emb = extract_embedding(seg_path)
        score = cosine_similarity(ref_emb, seg_emb).item()
        scores.append(score)
        print(f"{seg_path:<60} | {score:.4f}")

    except Exception as e:
        print(f"Ошибка в файле {seg_path}: {e}")

# 4. Вывод среднего значения
if scores:
    mean_score = sum(scores) / len(scores)
    print("-" * 75)
    print(f"{'СРЕДНЕЕ ЗНАЧЕНИЕ:':<60} | {mean_score:.4f}")



/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Файл                                                         | Похожесть 
---------------------------------------------------------------------------
/content/sentence_0.wav                                      | 0.9519
/content/sentence_1.wav                                      | 0.9008
/content/sentence_2.wav                                      | 0.8460
/content/sentence_3.wav                                      | 0.9441
/content/sentence_4.wav                                      | 0.9343
/content/sentence_5.wav                                      | 0.9272
/content/sentence_6.wav                                      | 0.9070
/content/sentence_7.wav                                      | 0.9350
/content/sentence_8.wav                                      | 0.9448
/content/sentence_9.wav                                      | 0.9624
---------------------------------------------------------------------------
СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.9254


Среднее значение получилось:: 0.9254

**Посчитаем похожесть со вторым референсом segment_0001.wav**

In [ ]:
# 1. Задаем пути
reference_file = '/content/coqui-ai-TTS/dataset/wavs/segment_0001.wav'

# Автоматически создаем список путей от sentence_0.wav до sentence_9.wav
candidate_files = [f'/content/sentence_{i}.wav' for i in range(10)]

# 2. Извлекаем эмбеддинг эталона
ref_emb = extract_embedding(reference_file)

scores = []

print(f"{'Файл':<60} | {'Похожесть':<10}")
print("-" * 75)

# 3. Сравниваем в цикле
for seg_path in candidate_files:
    # Проверяем, существует ли файл, чтобы код не упал с ошибкой
    if not os.path.exists(seg_path):
        print(f"Файл не найден: {seg_path}")
        continue

    try:
        seg_emb = extract_embedding(seg_path)
        score = cosine_similarity(ref_emb, seg_emb).item()
        scores.append(score)
        print(f"{seg_path:<60} | {score:.4f}")

    except Exception as e:
        print(f"Ошибка в файле {seg_path}: {e}")

# 4. Вывод среднего значения
if scores:
    mean_score = sum(scores) / len(scores)
    print("-" * 75)
    print(f"{'СРЕДНЕЕ ЗНАЧЕНИЕ:':<60} | {mean_score:.4f}")

Файл                                                         | Похожесть 
---------------------------------------------------------------------------
/content/sentence_0.wav                                      | 0.9460
/content/sentence_1.wav                                      | 0.8925
/content/sentence_2.wav                                      | 0.8662
/content/sentence_3.wav                                      | 0.9248
/content/sentence_4.wav                                      | 0.9328
/content/sentence_5.wav                                      | 0.9109
/content/sentence_6.wav                                      | 0.8865
/content/sentence_7.wav                                      | 0.9197
/content/sentence_8.wav                                      | 0.9275
/content/sentence_9.wav                                      | 0.9611
---------------------------------------------------------------------------
СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.9168


СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.9168

**Посчитаем похожесть с третьим референсом segment_0002.wav**

In [ ]:
# 1. Задаем пути
reference_file = '/content/coqui-ai-TTS/dataset/wavs/segment_0002.wav'

# Автоматически создаем список путей от sentence_0.wav до sentence_9.wav
candidate_files = [f'/content/sentence_{i}.wav' for i in range(10)]

# 2. Извлекаем эмбеддинг эталона
ref_emb = extract_embedding(reference_file)

scores = []

print(f"{'Файл':<60} | {'Похожесть':<10}")
print("-" * 75)

# 3. Сравниваем в цикле
for seg_path in candidate_files:
    # Проверяем, существует ли файл, чтобы код не упал с ошибкой
    if not os.path.exists(seg_path):
        print(f"Файл не найден: {seg_path}")
        continue

    try:
        seg_emb = extract_embedding(seg_path)
        score = cosine_similarity(ref_emb, seg_emb).item()
        scores.append(score)
        print(f"{seg_path:<60} | {score:.4f}")

    except Exception as e:
        print(f"Ошибка в файле {seg_path}: {e}")

# 4. Вывод среднего значения
if scores:
    mean_score = sum(scores) / len(scores)
    print("-" * 75)
    print(f"{'СРЕДНЕЕ ЗНАЧЕНИЕ:':<60} | {mean_score:.4f}")

Файл                                                         | Похожесть 
---------------------------------------------------------------------------
/content/sentence_0.wav                                      | 0.9723
/content/sentence_1.wav                                      | 0.9437
/content/sentence_2.wav                                      | 0.9320
/content/sentence_3.wav                                      | 0.9672
/content/sentence_4.wav                                      | 0.9532
/content/sentence_5.wav                                      | 0.9561
/content/sentence_6.wav                                      | 0.9340
/content/sentence_7.wav                                      | 0.9466
/content/sentence_8.wav                                      | 0.9214
/content/sentence_9.wav                                      | 0.9575
---------------------------------------------------------------------------
СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.9484


СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.9484

###**Модель zero-shot**

Посчитаем похожесть с первым референсом segment_0000.wav

In [ ]:
import os

# 1. Задаем пути
reference_file = '/content/coqui-ai-TTS/dataset/wavs/segment_0000.wav'

# Генерируем список путей: от 0_male.wav до 9_male.wav
candidate_files = [f'/content/generated_samples/sim_test/{i}_male.wav' for i in range(10)]

# 2. Извлекаем эмбеддинг эталона
ref_emb = extract_embedding(reference_file)

scores = []

print(f"{'Файл':<60} | {'Похожесть':<10}")
print("-" * 75)

# 3. Сравниваем в цикле
for seg_path in candidate_files:
    # Проверка на существование файла, чтобы избежать ошибок при расчете
    if not os.path.exists(seg_path):
        print(f"Файл не найден: {seg_path:<45} | Пропуск")
        continue

    try:
        seg_emb = extract_embedding(seg_path)
        score = cosine_similarity(ref_emb, seg_emb).item()
        scores.append(score)
        print(f"{seg_path:<60} | {score:.4f}")

    except Exception as e:
        print(f"Ошибка в файле {seg_path}: {e}")

# 4. Вывод среднего значения
if scores:
    mean_score = sum(scores) / len(scores)
    print("-" * 75)
    print(f"{'СРЕДНЕЕ ЗНАЧЕНИЕ:':<60} | {mean_score:.4f}")
else:
    print("Нет данных для расчета среднего значения.")


Файл                                                         | Похожесть 
---------------------------------------------------------------------------
/content/generated_samples/sim_test/0_male.wav               | 0.8758
/content/generated_samples/sim_test/1_male.wav               | 0.8919
/content/generated_samples/sim_test/2_male.wav               | 0.8156
/content/generated_samples/sim_test/3_male.wav               | 0.8546
/content/generated_samples/sim_test/4_male.wav               | 0.8915
/content/generated_samples/sim_test/5_male.wav               | 0.8666
/content/generated_samples/sim_test/6_male.wav               | 0.8950
/content/generated_samples/sim_test/7_male.wav               | 0.8541
/content/generated_samples/sim_test/8_male.wav               | 0.8766
/content/generated_samples/sim_test/9_male.wav               | 0.8846
---------------------------------------------------------------------------
СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.8706


СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.8706

In [ ]:
import os

# 1. Задаем пути
reference_file = '/content/coqui-ai-TTS/dataset/wavs/segment_0001.wav'

# Генерируем список путей: от 0_male.wav до 9_male.wav
candidate_files = [f'/content/generated_samples/sim_test/{i}_male.wav' for i in range(10)]

# 2. Извлекаем эмбеддинг эталона
ref_emb = extract_embedding(reference_file)

scores = []

print(f"{'Файл':<60} | {'Похожесть':<10}")
print("-" * 75)

# 3. Сравниваем в цикле
for seg_path in candidate_files:
    # Проверка на существование файла, чтобы избежать ошибок при расчете
    if not os.path.exists(seg_path):
        print(f"Файл не найден: {seg_path:<45} | Пропуск")
        continue

    try:
        seg_emb = extract_embedding(seg_path)
        score = cosine_similarity(ref_emb, seg_emb).item()
        scores.append(score)
        print(f"{seg_path:<60} | {score:.4f}")

    except Exception as e:
        print(f"Ошибка в файле {seg_path}: {e}")

# 4. Вывод среднего значения
if scores:
    mean_score = sum(scores) / len(scores)
    print("-" * 75)
    print(f"{'СРЕДНЕЕ ЗНАЧЕНИЕ:':<60} | {mean_score:.4f}")
else:
    print("Нет данных для расчета среднего значения.")

Файл                                                         | Похожесть 
---------------------------------------------------------------------------
/content/generated_samples/sim_test/0_male.wav               | 0.8914
/content/generated_samples/sim_test/1_male.wav               | 0.8400
/content/generated_samples/sim_test/2_male.wav               | 0.8683
/content/generated_samples/sim_test/3_male.wav               | 0.8683
/content/generated_samples/sim_test/4_male.wav               | 0.8999
/content/generated_samples/sim_test/5_male.wav               | 0.8467
/content/generated_samples/sim_test/6_male.wav               | 0.9145
/content/generated_samples/sim_test/7_male.wav               | 0.8518
/content/generated_samples/sim_test/8_male.wav               | 0.8566
/content/generated_samples/sim_test/9_male.wav               | 0.9173
---------------------------------------------------------------------------
СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.8755


СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.8755

In [ ]:
import os

# 1. Задаем пути
reference_file = '/content/coqui-ai-TTS/dataset/wavs/segment_0002.wav'

# Генерируем список путей: от 0_male.wav до 9_male.wav
candidate_files = [f'/content/generated_samples/sim_test/{i}_male.wav' for i in range(10)]

# 2. Извлекаем эмбеддинг эталона
ref_emb = extract_embedding(reference_file)

scores = []

print(f"{'Файл':<60} | {'Похожесть':<10}")
print("-" * 75)

# 3. Сравниваем в цикле
for seg_path in candidate_files:
    # Проверка на существование файла, чтобы избежать ошибок при расчете
    if not os.path.exists(seg_path):
        print(f"Файл не найден: {seg_path:<45} | Пропуск")
        continue

    try:
        seg_emb = extract_embedding(seg_path)
        score = cosine_similarity(ref_emb, seg_emb).item()
        scores.append(score)
        print(f"{seg_path:<60} | {score:.4f}")

    except Exception as e:
        print(f"Ошибка в файле {seg_path}: {e}")

# 4. Вывод среднего значения
if scores:
    mean_score = sum(scores) / len(scores)
    print("-" * 75)
    print(f"{'СРЕДНЕЕ ЗНАЧЕНИЕ:':<60} | {mean_score:.4f}")
else:
    print("Нет данных для расчета среднего значения.")

Файл                                                         | Похожесть 
---------------------------------------------------------------------------
/content/generated_samples/sim_test/0_male.wav               | 0.8932
/content/generated_samples/sim_test/1_male.wav               | 0.8691
/content/generated_samples/sim_test/2_male.wav               | 0.8498
/content/generated_samples/sim_test/3_male.wav               | 0.8743
/content/generated_samples/sim_test/4_male.wav               | 0.9071
/content/generated_samples/sim_test/5_male.wav               | 0.8985
/content/generated_samples/sim_test/6_male.wav               | 0.9201
/content/generated_samples/sim_test/7_male.wav               | 0.8651
/content/generated_samples/sim_test/8_male.wav               | 0.8903
/content/generated_samples/sim_test/9_male.wav               | 0.8918
---------------------------------------------------------------------------
СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.8859


СРЕДНЕЕ ЗНАЧЕНИЕ:                                            | 0.8859

### **Примеры аудиозаписей:**

Оригинальный голос:

In [ ]:
display(Audio('/content/coqui-ai-TTS/dataset/wavs/segment_0003.wav'))

Клоннированная аудиозапись zero-shot:

In [ ]:
Audio('/content/generated_samples/sim_test/3_male.wav')

Клонированная аудиозапись(модель, дообученная на одной минуте из другого блокнота)

In [ ]:
Audio('/content/sentence_3.wav')

Клонированная аудиозапись(модель, дообученная на 10ти минутах)

In [ ]:
Audio('/content/sentence_3.wav')

###**Результаты**

***Из-за ограничений на вычислительные ресурсы пришлось доделывать обучение на одной минуте во втором блокноте он также прилагается.***

**В качестве референсных в обоих блоконтах брались одни и те же записи**

**Первый референс:**

Zero shot: 0.8706

1-min: 0.8984

10-min: 0.9254

**Второй референс:**

Zero shot: 0.8755

1-min: 0.9160

10-min: 0.9168

**Третий референс:**

Zero shot: 0.8859

1-min: 0.9239

10-min: 0.9484

**По сгенерированным аудиозаписям слышно, что с дообучением на 1 минуте, а затем на 10ти минутах тембр и "текстура" голоса приближаются к оригиналу, что так же видно и на объективных метриках.**